# Phase 3 — Feature Engineering & Stratified Split: Validation Layer

**Status:** ✅ Completed (pipeline v1.0)  
**Authoritative Script:** `ml_service/src/feature_engineer.py`  
**Source Artifacts:** `artifacts/splits/*` (Parquet + Pickle)  

---

## Executive Summary
Phase 3 performs **3-pass streaming feature engineering** over the 2,520,798 clean flows:
1. **Pass 1:** Scaler fit on training indices only (`StandardScaler(with_mean=True, with_std=True)`).
2. **Pass 2:** Transform float32 features and stream write `X_train_v1.parquet` (2,016,638 rows) and `X_test_v1.parquet` (504,160 rows).
3. **Pass 3:** Encode and write integer target labels for both 15-class multiclass and binary intrusion detection.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns

root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from config.paths import Paths

with open(Paths.SPLITS_DIR / 'split_summary_v1.json', 'r', encoding='utf-8') as f:
    summary = json.load(f)

print(f"Phase:            {summary['phase']}")
print(f"Pipeline Version: {summary['pipeline_version']}")
print(f"Train Rows (80%): {summary['row_counts']['train']:,}")
print(f"Test Rows (20%):  {summary['row_counts']['test']:,}")
print(f"Feature Count:    {len(summary['features']['scaled_feature_names'])}")
print(f"Peak Resident RAM:{summary['memory']['peak_rss_mb']:.2f} MB (Budget: {summary['memory']['memory_limit_mb']} MB)")

## 1. Stratification Parity Across Train and Test Sets
To ensure unbiased evaluation, the 80/20 split enforces strict stratification across all 15 classes, preserving minority samples down to Heartbleed (9 in train, 2 in test) and SQL Injection (17 in train, 4 in test).

In [ ]:
train_dist = summary['class_distribution']['train']
test_dist  = summary['class_distribution']['test']
classes = summary['label_encoding']['category_classes']

strat_rows = []
for cls in classes:
    n_tr = train_dist[cls]
    n_te = test_dist[cls]
    pct_tr = (n_tr / summary['row_counts']['train']) * 100
    pct_te = (n_te / summary['row_counts']['test']) * 100
    strat_rows.append({
        'Category': cls,
        'Train Count': n_tr,
        'Train %': pct_tr,
        'Test Count': n_te,
        'Test %': pct_te,
        'Delta %': abs(pct_tr - pct_te)
    })

df_strat = pd.DataFrame(strat_rows)
df_strat.style.format({
    'Train Count': '{:,}', 'Test Count': '{:,}',
    'Train %': '{:.4f}%', 'Test %': '{:.4f}%', 'Delta %': '{:.6f}%'
})

## 2. Feature Schema & Column Role Decomposition
Separating raw telemetry into model input features, correlation identifiers, and training labels:

In [ ]:
schema_overview = {
    'Column Role': [
        'ML Continuous Features (Scaled float32)',
        'Downstream Identifier (Preserved for Event Correlation)',
        'Redundant Duplicate Column (Purged)',
        'Classification Target Label'
    ],
    'Count': [76, 1, 1, 1],
    'Representative Examples': [
        'Flow Duration, Flow Bytes/s, Fwd Packet Length Mean, Init_Win_bytes_forward',
        'Destination Port (Crucial for patator & portscan grouping)',
        'Fwd Header Length.1 (exact duplicate of Fwd Header Length)',
        'Label (Encoded into 15 multiclass integers + binary 0/1)'
    ]
}
pd.DataFrame(schema_overview)

## 3. Standard Scaler Parameter Inspection
Inspecting the fitted StandardScaler parameters (`artifacts/splits/scaler_v1.pkl`) on representative features:

In [ ]:
scaler = joblib.load(Paths.split('scaler_v1.pkl'))
feat_names = summary['features']['scaled_feature_names']

scaler_df = pd.DataFrame({
    'Feature Name': feat_names[:15],
    'Training Mean (μ)': scaler.mean_[:15],
    'Training Std (σ)': np.sqrt(scaler.var_[:15])
})
scaler_df.style.format({'Training Mean (μ)': '{:,.4f}', 'Training Std (σ)': '{:,.4f}'})